# Data Collation: Downloading the SalmoSapro Dataset

This notebook provides the code to download the image dataset used in this study. The image URLs and their corresponding classifications are sourced from the publicly available metadata catalog on Zenodo.

**Reference:**
Olsen, A. S., Cook, N., & Perkins, S. E. (2025). *SalmoSapro Metadata Catalog: A Cross-Platform Index of Salmonid Images with Saprolegnia Classifications* (Version 1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.15097673

### Important Note on Data Availability

As stated in the manuscript, the full dataset used for training consists of both publicly sourced images (downloaded via this notebook) and images provided under specific data sharing agreements.

**Data Availability Statement:**
> The complete code for data processing, model training, and analysis used in this study is publicly available on GitHub at [Link to Your GitHub Repository]. The metadata for all images sourced from public repositories (iNaturalist, Flickr, GBIF, Wikimedia Commons), including the URLs required to download these images, is available on Zenodo (Olsen et al., 2025; DOI: 10.5281/zenodo.15097672). A portion of the ‘Saprolegnia spp.’ class images was provided directly by stakeholders. These images were shared with us for analysis under specific data sharing agreements with the original contributors and are therefore not publicly available for redistribution.

Therefore, running this notebook will download the **publicly available portion** of the dataset.

### Step 1: Download the Metadata

Before running this notebook, you must first download the metadata file from the Zenodo repository.

1. Go to the Zenodo record: [https://zenodo.org/records/15097673](https://zenodo.org/records/15097673)
2. Download the `sapro_final_database.csv` file.
3. Place the downloaded CSV file in the same directory as this notebook.


### Step 2: Setup and Imports

This step imports the necessary Python libraries for this task. You will need `pandas`, `requests`, and `tqdm`. If these are not installed, you can install them by running `pip install pandas requests tqdm` in your terminal after activating your environment.


In [ ]:
import pandas as pd
import requests
from pathlib import Path
import os
from tqdm.auto import tqdm
import time


### Step 3: Configure Paths and Load Metadata

Here, we define where the images will be saved and load the metadata file into a pandas DataFrame.


In [ ]:
# Configuration
METADATA_FILE = 'sapro_final_database.csv'
DOWNLOAD_DIR = Path('../data/downloaded_dataset') # All downloaded images will go here
SUBSET_DIR = Path('../data/subsets') # All symlinked subsets will go here

# Create directories if they don't exist
DOWNLOAD_DIR.mkdir(exist_ok=True)
SUBSET_DIR.mkdir(exist_ok=True)


# Load the metadata
try:
    df = pd.read_csv(METADATA_FILE)
    print(f"Successfully loaded metadata for {len(df)} images.")
    print("Class distribution:")
    print(df['saprolegnia'].value_counts())
except FileNotFoundError:
    print(f"Error: Metadata file '{METADATA_FILE}' not found.")
    print("Please ensure you have downloaded it from Zenodo and placed it in the correct directory.")
    df = None


### Step 4: Download the Images

This is the main part of the notebook. The code below will:
1. Create the necessary output directories (`healthy` and `sapro`).
2. Iterate through each row of the metadata.
3. Attempt to download the image from the specified `image_url`.
4. Save the image to the correct directory based on its `saprolegnia` label.
5. Use the last part of the URL as the filename.

**Note:** This process may take a considerable amount of time depending on your internet connection and the number of images. Some URLs may also be broken or no longer accessible, which is common with web-sourced data.


In [ ]:
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
import re

def download_images(metadata_df, download_dir):
    if metadata_df is None:
        print("Metadata not loaded. Cannot proceed with download.")
        return

    # --- Pre-flight Check: Verify required columns ---
    required_cols = ['saprolegnia', 'image_url', 'source']
    if not all(col in metadata_df.columns for col in required_cols):
        print(f"Error: Metadata file is missing one or more required columns. Expected: {required_cols}")
        print(f"Found columns: {metadata_df.columns.tolist()}")
        return

    # Create base output directory
    download_dir.mkdir(exist_ok=True)
    
    # Create class subdirectories
    class_map = {'yes': 'sapro', 'no': 'healthy'}
    for new_dir in class_map.values():
        (download_dir / new_dir).mkdir(exist_ok=True)

    print(f"Images will be saved in: {download_dir.resolve()}\\n")
    
    # --- Setup robust session for requests ---
    session = requests.Session()
    retry_strategy = Retry(
        total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36'}
    
    downloaded_metadata = []
    skipped_labels = 0
    failed_content_type = 0
    inat_transformed_count = 0
    
    for index, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Downloading Images"):
        status = "Skipped"
        final_filename = None
        try:
            label = row['saprolegnia']
            url = row['image_url']
            download_url = url # Default to original URL
            
            class_dir = class_map.get(str(label).lower())
            if class_dir is None:
                skipped_labels += 1
                continue

            # --- Extract Photo ID and Source for Filename ---
            photo_id = re.findall(r'\d+', url)
            photo_id = photo_id[-1] if photo_id else "unknown"
            source = str(row.get('source', 'unknown')).replace(' ', '_')

            # --- Handle iNaturalist URLs ---
            if 'inaturalist.org' in url and photo_id != "unknown":
                # Construct a direct URL to the 'original' image file on the S3 bucket
                new_download_url = f"https://inaturalist-open-data.s3.amazonaws.com/photos/{photo_id}/original.jpg"
                if url != new_download_url:
                    inat_transformed_count += 1
                download_url = new_download_url

            # --- Generate a descriptive and unique filename ---
            file_extension = '.jpg' # Default to .jpg for consistency, especially for iNat
            original_ext = os.path.splitext(url.split('?')[0])[-1]
            if original_ext and len(original_ext) <= 5:
                file_extension = original_ext
            
            final_filename = f"{index}_{source}_{photo_id}{file_extension}".replace(' ', '_')


            save_path = download_dir / class_dir / final_filename
            if save_path.exists():
                status = "Success: Already exists"
            else:
                response = session.get(download_url, headers=headers, timeout=20, stream=True)
                response.raise_for_status()

                content_type = response.headers.get('Content-Type', '')
                if 'image' not in content_type:
                    failed_content_type += 1
                    status = f"Failed: Invalid Content-Type ('{content_type}') from URL: {download_url}"
                    continue

                with open(save_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                status = "Success"

        except requests.exceptions.RequestException as e:
            if e.response and 400 <= e.response.status_code < 500 and e.response.status_code != 429:
                 status = f"Failed: Client Error {e.response.status_code}"
            else:
                status = f"Failed: Request Error"
        except Exception as e:
            status = f"Failed: General Error"
            
        if status.startswith("Success"):
            new_row = row.to_dict()
            new_row['download_status'] = status
            new_row['filename'] = final_filename
            downloaded_metadata.append(new_row)

    print("\\n--- Download Complete ---")
    if inat_transformed_count > 0:
        print(f"Transformed {inat_transformed_count} iNaturalist page URLs into direct image links.")
    if skipped_labels > 0:
        print(f"Skipped {skipped_labels} rows due to unrecognized labels in the 'saprolegnia' column.")
    if failed_content_type > 0:
        print(f"Skipped {failed_content_type} downloads due to invalid content type (e.g., HTML page instead of image).")

    if downloaded_metadata:
        downloaded_df = pd.DataFrame(downloaded_metadata)
        output_metadata_path = download_dir / 'downloaded_images_metadata.csv'
        downloaded_df.to_csv(output_metadata_path, index=False)
        print(f"\\nSuccessfully processed {len(downloaded_df)} images (new or existing).")
        print(f"A metadata file for these images has been saved to: {output_metadata_path.resolve()}")
    else:
        print("\\nNo images were successfully downloaded or found.")


# Run the download process
download_images(df, DOWNLOAD_DIR)

### Step 5: Create Taxonomic Subsets

This section contains the code to generate the specific taxonomic subsets used for the analyses in the paper. It uses the `taxonomic_info.csv` file to link the photographs to their respective genera (*Salmo*, *Oncorhynchus*, etc.).

**Prerequisite:** Ensure the `taxonomic_info.csv` file is present in the data folder, or change the path for this file.


In [ ]:
import shutil

def create_subsets(metadata_df, taxo_info_df, base_dataset_dir, subset_dir):
    if metadata_df is None:
        print("Metadata not loaded. Cannot create subsets.")
        return
    if taxo_info_df is None:
        print("Taxonomy info not loaded. Cannot create subsets.")
        return

    # --- 1. Robustly merge metadata with taxonomic info to find the Genus ---
    print("Preparing taxonomic data for merging...")
    
    # --- Robustly process the taxonomy file ---
    taxo_info_df['Family'] = taxo_info_df['Family'].ffill()
    taxo_info_df['Subfamily'] = taxo_info_df['Subfamily'].ffill()
    taxo_info_df['Genus'] = taxo_info_df['Genus'].ffill()

    # --- Create lowercase versions for case-insensitive matching ---
    taxo_info_df['genus_lower'] = taxo_info_df['Genus'].str.lower()
    taxo_info_df['species_lower'] = taxo_info_df['Species'].str.lower()

    # Create mappings, dropping duplicates to ensure a 1-to-1 map and prevent ambiguity
    species_map_df = taxo_info_df.dropna(subset=['species_lower']).drop_duplicates(subset=['species_lower'], keep='first')
    species_to_genus = species_map_df.set_index('species_lower')['Genus']
    
    genus_map_df = taxo_info_df.dropna(subset=['genus_lower']).drop_duplicates(subset=['genus_lower'], keep='first')
    genus_lower_to_genus = genus_map_df.set_index('genus_lower')['Genus']
    valid_genera_lower = set(genus_lower_to_genus.index)
    

    def find_genus(taxa_name):
        if pd.isna(taxa_name):
            return None
        
        taxa_lower = str(taxa_name).lower()
        
        # Case 1: Direct match with a species name
        genus = species_to_genus.get(taxa_lower)
        if isinstance(genus, str): return genus
        
        # Case 2: Direct match with a genus name
        if taxa_lower in valid_genera_lower:
            return genus_lower_to_genus.get(taxa_lower)
            
        # Case 3: Match the first part of a binomial name (e.g., "salmo trutta")
        parts = taxa_lower.split()
        if len(parts) > 1 and parts[0] in valid_genera_lower:
            return genus_lower_to_genus.get(parts[0])
            
        return None # No match found

    full_df = metadata_df.copy()
    full_df['Genus'] = full_df['taxa'].apply(find_genus)
    
    # --- Report on matching success ---
    matched_count = full_df['Genus'].notna().sum()
    total_count = len(full_df)
    print(f"Successfully merged metadata with taxonomy. {matched_count} of {total_count} rows matched a genus.")
    if matched_count < total_count:
        unmatched_taxa = full_df[full_df['Genus'].isna()]['taxa'].value_counts()
        print(f"\\nCould not match {len(unmatched_taxa)} unique taxa names:")
        print(unmatched_taxa.head(10)) # Print top 10 unmatched
    
    full_df['saprolegnia'] = pd.Categorical(full_df['saprolegnia'], categories=['yes', 'no'], ordered=False)
    
    # --- 2. Define and create dataset subsets ---
    
    subset_dir.mkdir(exist_ok=True)
    class_map = {'yes': 'sapro', 'no': 'healthy'}

    # Helper function to create symlinks
    def create_symlinks_for_subset(subset_df, subset_name):
        subset_path = subset_dir / subset_name
        print(f"\\nCreating subset: {subset_name} ({len(subset_df)} images)")
        
        if subset_path.exists():
            shutil.rmtree(subset_path)
        subset_path.mkdir()
        (subset_path / 'healthy').mkdir()
        (subset_path / 'sapro').mkdir()
        
        for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc=f"Creating '{subset_name}'"):
            class_dir = class_map.get(str(row['saprolegnia']).lower())
            
            if 'filename' in row and pd.notna(row['filename']):
                filename = row['filename']
            else:
                url = row['image_url']
                photo_id = re.findall(r'\d+', url)
                photo_id = photo_id[-1] if photo_id else "unknown"
                source = str(row.get('source', 'unknown')).replace(' ', '_')
                file_extension = '.jpg'
                original_ext = os.path.splitext(url.split('?')[0])[-1]
                if original_ext and len(original_ext) <= 5:
                    file_extension = original_ext
                filename = f"{row.name}_{source}_{photo_id}{file_extension}".replace(' ', '_')

            original_path = base_dataset_dir / class_dir / filename
            link_path = subset_path / class_dir / filename

            if original_path.exists():
                os.symlink(original_path.resolve(), link_path)

    # --- Subset A: All photographs (already exists) ---
    print("\\nSubset 'All photographs' is the base downloaded dataset.")

    # --- Subset B: Taxa with photographs in both classes ---
    class_counts = full_df.groupby('taxa')['saprolegnia'].value_counts().unstack(fill_value=0)
    taxa_in_both = class_counts[(class_counts['yes'] > 0) & (class_counts['no'] > 0)].index
    subset_b_df = full_df[full_df['taxa'].isin(taxa_in_both)]
    create_symlinks_for_subset(subset_b_df, 'B_taxa_in_both_classes')

    # --- Subset C: Taxa with >=10 photographs in both classes ---
    taxa_10_in_both = class_counts[(class_counts['yes'] >= 10) & (class_counts['no'] >= 10)].index
    subset_c_df = full_df[full_df['taxa'].isin(taxa_10_in_both)]
    create_symlinks_for_subset(subset_c_df, 'C_taxa_geq_10_in_both')

    # --- Subset D: Oncorhynchus, >=10 photographs in both classes (per species) ---
    onco_df = full_df[full_df['Genus'] == 'Oncorhynchus'].copy()
    onco_class_counts = onco_df.groupby('taxa')['saprolegnia'].value_counts().unstack(fill_value=0)
    onco_taxa_10_in_both = onco_class_counts[(onco_class_counts['yes'] >= 10) & (onco_class_counts['no'] >= 10)].index
    subset_d_df = onco_df[onco_df['taxa'].isin(onco_taxa_10_in_both)]
    create_symlinks_for_subset(subset_d_df, 'D_oncorhynchus_geq_10_in_both')
    
    # --- Subset E: Salmo, >=10 photographs in both classes (per species) ---
    salmo_df = full_df[full_df['Genus'] == 'Salmo'].copy()
    salmo_class_counts = salmo_df.groupby('taxa')['saprolegnia'].value_counts().unstack(fill_value=0)
    salmo_taxa_10_in_both = salmo_class_counts[(salmo_class_counts['yes'] >= 10) & (salmo_class_counts['no'] >= 10)].index
    subset_e_df = salmo_df[salmo_df['taxa'].isin(salmo_taxa_10_in_both)]
    create_symlinks_for_subset(subset_e_df, 'E_salmo_geq_10_in_both')


# --- Run the subset creation ---
# Use the new downloaded metadata if available, otherwise fallback to original df
try:
    downloaded_metadata_path = DOWNLOAD_DIR / 'downloaded_images_metadata.csv'
    if downloaded_metadata_path.exists():
        print(f"Using downloaded metadata from: {downloaded_metadata_path}")
        df_for_subsets = pd.read_csv(downloaded_metadata_path)
    else:
        print("Using original metadata for subset creation.")
        df_for_subsets = df

    taxo_df = pd.read_csv('../data/taxonomic_info.csv')
    create_subsets(df_for_subsets, taxo_df, DOWNLOAD_DIR, SUBSET_DIR)
except FileNotFoundError as e:
    print(f"\\nError: A required file was not found. Please check paths. Details: {e}")
except Exception as e:
    print(f"\\nAn error occurred during subset creation: {e}")